In [5]:
from preprocessing import * 

NameError: name 'SimpleImputer' is not defined

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
import math
from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.impute import SimpleImputer
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
from sklearn.metrics import classification_report
from sklearn.decomposition import PCA
from imblearn.combine import SMOTETomek
from sklearn import metrics

In [118]:
# MOVED TO MAIN
train_raw_df = pd.read_csv("drive/MyDrive/MLcourse/customer_segmentation_train.csv")
train_raw_df.head()

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A


In [119]:
train_raw_df['Segmentation'].unique()

array(['D', 'A', 'B', 'C'], dtype=object)

# Main

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [3]:
train_raw_df = pd.read_csv("customer_segmentation_train.csv")

In [223]:
columns_list = train_raw_df.columns.to_list()

In [4]:
X_train, X_test = split_train_test(train_raw_df, 'Segmentation')

NameError: name 'split_train_test' is not defined

In [198]:
X_train

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
917,465905,Female,No,32,Yes,Artist,9.0,Low,1.0,Cat_6,A
3398,462903,Male,Yes,72,Yes,Entertainment,NaN,Average,2.0,Cat_6,B
2045,467901,Female,No,33,Yes,Entertainment,1.0,Low,4.0,Cat_6,B
8060,463613,Female,Yes,48,Yes,Artist,0.0,Average,6.0,Cat_6,A
4604,459859,Female,Yes,28,No,Doctor,9.0,Low,1.0,Cat_7,A
...,...,...,...,...,...,...,...,...,...,...,...
3822,463101,Female,No,27,No,Homemaker,8.0,Low,1.0,Cat_6,D
5864,467844,Male,No,37,Yes,Healthcare,0.0,Low,2.0,Cat_6,D
3589,460706,Female,No,27,No,Engineer,6.0,Low,6.0,Cat_4,D
1489,464339,Male,No,26,No,Artist,0.0,Low,2.0,Cat_6,D


In [225]:
X_train, y_train = extract_target(X_train, "Segmentation")

In [137]:
X_train

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1
917,465905,Female,No,32,Yes,Artist,9.0,Low,1.0,Cat_6
3398,462903,Male,Yes,72,Yes,Entertainment,NaN,Average,2.0,Cat_6
2045,467901,Female,No,33,Yes,Entertainment,1.0,Low,4.0,Cat_6
8060,463613,Female,Yes,48,Yes,Artist,0.0,Average,6.0,Cat_6
4604,459859,Female,Yes,28,No,Doctor,9.0,Low,1.0,Cat_7
...,...,...,...,...,...,...,...,...,...,...
3822,463101,Female,No,27,No,Homemaker,8.0,Low,1.0,Cat_6
5864,467844,Male,No,37,Yes,Healthcare,0.0,Low,2.0,Cat_6
3589,460706,Female,No,27,No,Engineer,6.0,Low,6.0,Cat_4
1489,464339,Male,No,26,No,Artist,0.0,Low,2.0,Cat_6


In [226]:
# use LabelEncoding to encode the target variable
target_encoder = LabelEncoder()
y_train = target_encoder.fit_transform(y_train)

In [177]:
y_train

array([0, 1, 1, ..., 3, 3, 0])

In [202]:
cols_with_na, cols_to_delete = detect_missing_values_cols(X_train)

In [227]:
X_train = drop_columns_splits(X_train, "ID")

In [228]:
X_train

,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1
917,Female,No,32,Yes,Artist,9.0,Low,1.0,Cat_6
3398,Male,Yes,72,Yes,Entertainment,NaN,Average,2.0,Cat_6
2045,Female,No,33,Yes,Entertainment,1.0,Low,4.0,Cat_6
8060,Female,Yes,48,Yes,Artist,0.0,Average,6.0,Cat_6
4604,Female,Yes,28,No,Doctor,9.0,Low,1.0,Cat_7
...,...,...,...,...,...,...,...,...,...
3822,Female,No,27,No,Homemaker,8.0,Low,1.0,Cat_6
5864,Male,No,37,Yes,Healthcare,0.0,Low,2.0,Cat_6
3589,Female,No,27,No,Engineer,6.0,Low,6.0,Cat_4
1489,Male,No,26,No,Artist,0.0,Low,2.0,Cat_6


In [229]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6454 entries, 917 to 2661
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Gender           6454 non-null   object 
 1   Ever_Married     6343 non-null   object 
 2   Age              6454 non-null   int64  
 3   Graduated        6395 non-null   object 
 4   Profession       6348 non-null   object 
 5   Work_Experience  5808 non-null   float64
 6   Spending_Score   6454 non-null   object 
 7   Family_Size      6190 non-null   float64
 8   Var_1            6394 non-null   object 
dtypes: float64(2), int64(1), object(6)
memory usage: 504.2+ KB


In [230]:
X_train.isna().sum()

,0
Gender,0
Ever_Married,111
Age,0
Graduated,59
Profession,106
Work_Experience,646
Spending_Score,0
Family_Size,264
Var_1,60


In [231]:
X_train, numerical_imputer, categorical_imputer = impute_miss_values(X_train)

In [184]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6454 entries, 917 to 2661
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Gender           6454 non-null   object 
 1   Ever_Married     6454 non-null   object 
 2   Age              6454 non-null   float64
 3   Graduated        6454 non-null   object 
 4   Profession       6454 non-null   object 
 5   Work_Experience  6454 non-null   float64
 6   Spending_Score   6454 non-null   object 
 7   Family_Size      6454 non-null   float64
 8   Var_1            6454 non-null   object 
dtypes: float64(3), object(6)
memory usage: 504.2+ KB


In [232]:
X_train.isna().sum()

,0
Gender,0
Ever_Married,0
Age,0
Graduated,0
Profession,0
Work_Experience,0
Spending_Score,0
Family_Size,0
Var_1,0


In [186]:
X_train

,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1
917,Female,No,32.0,Yes,Artist,9.0,Low,1.0,Cat_6
3398,Male,Yes,72.0,Yes,Entertainment,1.0,Average,2.0,Cat_6
2045,Female,No,33.0,Yes,Entertainment,1.0,Low,4.0,Cat_6
8060,Female,Yes,48.0,Yes,Artist,0.0,Average,6.0,Cat_6
4604,Female,Yes,28.0,No,Doctor,9.0,Low,1.0,Cat_7
...,...,...,...,...,...,...,...,...,...
3822,Female,No,27.0,No,Homemaker,8.0,Low,1.0,Cat_6
5864,Male,No,37.0,Yes,Healthcare,0.0,Low,2.0,Cat_6
3589,Female,No,27.0,No,Engineer,6.0,Low,6.0,Cat_4
1489,Male,No,26.0,No,Artist,0.0,Low,2.0,Cat_6


In [233]:
X_train, encoder, categorical_features = encode_splits_categories(X_train)

In [234]:
X_train

,Age,Work_Experience,Family_Size,Gender_Female,Gender_Male,Ever_Married_No,Ever_Married_Yes,Graduated_No,Graduated_Yes,Profession_Artist,...,Spending_Score_Average,Spending_Score_High,Spending_Score_Low,Var_1_Cat_1,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7
917,32.0,9.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3398,72.0,1.0,2.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2045,33.0,1.0,4.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8060,48.0,0.0,6.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4604,28.0,9.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3822,27.0,8.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
5864,37.0,0.0,2.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3589,27.0,6.0,6.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1489,26.0,0.0,2.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [213]:
# scale: didn't use scale_numerical_cols() , will refactor later

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train[numerical_features])

In [214]:
X_train

array([[0.1971831 , 0.64285714, 0.        , ..., 0.        , 1.        ,
        0.        ],
       [0.76056338, 0.07142857, 0.125     , ..., 0.        , 1.        ,
        0.        ],
       [0.21126761, 0.07142857, 0.375     , ..., 0.        , 1.        ,
        0.        ],
       ...,
       [0.12676056, 0.42857143, 0.625     , ..., 0.        , 0.        ,
        0.        ],
       [0.11267606, 0.        , 0.125     , ..., 0.        , 1.        ,
        0.        ],
       [0.26760563, 0.        , 0.25      , ..., 0.        , 1.        ,
        0.        ]])

In [89]:
# reduce the dimensionality of input features: find the direction of maximum variance
"""
THIS STEP IS REMOVED FOR NOW BECAUSE OF THE SMOTENC INDEX ENCODING OF CATEGORIAL VARIABLES
pca = PCA(n_components=2)
pca.fit(X_train)
X_train_pca = pca.transform(X_train)

# for test set: X_test_pca = pca.transform(X_test)


# optimal number of principal components = explained variance
explained_var = pca.explained_variance_ratio_
cum_explained = explained_var.cumsum()
plt.figure(figsize=(7,4))
plt.plot(range(1, len(explained_var)+1), explained_var, marker="o")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.xticks(range(1, len(explained_var)+1))
plt.grid(True)
plt.show()
"""

In [ ]:
"""
LOAD THE TEST SET and preprocess
"""

In [154]:
X_test

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
6862,462588,Male,Yes,41,No,Entertainment,12.0,Average,3.0,Cat_4,A
3396,459767,Male,No,21,No,Healthcare,4.0,Low,4.0,Cat_6,D
6302,463475,Female,Yes,36,Yes,Doctor,0.0,Low,1.0,Cat_6,B
3329,465993,Female,No,27,No,Engineer,0.0,Low,2.0,Cat_6,D
7901,459016,Female,No,18,No,Healthcare,0.0,Low,6.0,Cat_6,D
...,...,...,...,...,...,...,...,...,...,...,...
6635,462299,Female,No,33,No,Healthcare,7.0,Low,6.0,Cat_6,C
5128,463891,Male,Yes,75,Yes,Lawyer,1.0,Low,1.0,Cat_6,A
2289,460046,Female,Yes,49,Yes,Artist,3.0,Low,2.0,Cat_6,B
5285,465211,Male,No,43,No,Artist,6.0,Low,2.0,Cat_6,B


In [157]:
X_test['Segmentation'].value_counts()

,count
Segmentation,
D,454
A,394
C,394
B,372


In [215]:
X_test = drop_columns_splits(X_test, "ID")

In [216]:
X_test, y_test = extract_target(X_test, "Segmentation")

In [217]:
# LabelEncoder for target
y_test = target_encoder.transform(y_test)
y_test

array([0, 3, 1, ..., 1, 1, 2])

In [218]:
X_test.isna().sum()

,0
Gender,0
Ever_Married,29
Age,0
Graduated,19
Profession,18
Work_Experience,183
Spending_Score,0
Family_Size,71
Var_1,16


In [219]:
# impute missing values

X_test[numerical_features] = numerical_imputer.transform(X_test[numerical_features])
X_test[categorical_features] = categorical_imputer.transform(X_test[categorical_features])

KeyError: "['Gender_Female', 'Gender_Male', 'Ever_Married_No', 'Ever_Married_Yes', 'Graduated_No', 'Graduated_Yes', 'Profession_Artist', 'Profession_Doctor', 'Profession_Engineer', 'Profession_Entertainment', 'Profession_Executive', 'Profession_Healthcare', 'Profession_Homemaker', 'Profession_Lawyer', 'Profession_Marketing', 'Spending_Score_Average', 'Spending_Score_High', 'Spending_Score_Low', 'Var_1_Cat_1', 'Var_1_Cat_2', 'Var_1_Cat_3', 'Var_1_Cat_4', 'Var_1_Cat_5', 'Var_1_Cat_6', 'Var_1_Cat_7'] not in index"

In [164]:
X_test.isna().sum()

,0
Gender,0
Ever_Married,0
Age,0
Graduated,0
Profession,0
Work_Experience,0
Spending_Score,0
Family_Size,0
Var_1,0


In [165]:
X_test = encoder.transform(X_test[categorical_features])
X_test

array([[0., 1., 0., ..., 0., 0., 0.],
       [0., 1., 1., ..., 0., 1., 0.],
       [1., 0., 0., ..., 0., 1., 0.],
       ...,
       [1., 0., 0., ..., 0., 1., 0.],
       [0., 1., 1., ..., 0., 1., 0.],
       [0., 1., 0., ..., 0., 0., 0.]])

In [194]:
X_test = scaler.transform(X_test[numerical_features])

KeyError: "['Gender_Female', 'Gender_Male', 'Ever_Married_No', 'Ever_Married_Yes', 'Graduated_No', 'Graduated_Yes', 'Profession_Artist', 'Profession_Doctor', 'Profession_Engineer', 'Profession_Entertainment', 'Profession_Executive', 'Profession_Healthcare', 'Profession_Homemaker', 'Profession_Lawyer', 'Profession_Marketing', 'Spending_Score_Average', 'Spending_Score_High', 'Spending_Score_Low', 'Var_1_Cat_1', 'Var_1_Cat_2', 'Var_1_Cat_3', 'Var_1_Cat_4', 'Var_1_Cat_5', 'Var_1_Cat_6', 'Var_1_Cat_7'] not in index"

In [117]:
# define index of categorical features
# categorical_features_indices
type(X_train)

numpy.ndarray

**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [91]:
"""
There should be some work around done with how you do SMOTE with only numerical features and how you combine them with
"""
smote = SMOTE(random_state=0)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

"\nclf_smotetomek = SVC(kernel='linear',probability=True)\nclf_smotetomek.fit(X_train_smotetomek, y_train_smotetomek)\n"

In [ ]:
# oversampling with SMOTENC on all features with SMOTENC
"""
SMOTENC has attribute 'CATEGORICAL_FEATURES' = you should pass the index of categorical features, not the list of features
"""

smotenc = SMOTENC(categorical_features=categorical_features, random_state=42)
X_train_smotenc, y_train_smotenc = smotenc.fit_resample(X_train, y_train)

In [ ]:
# combining oversampling + undersampling with Smote-tomek

smotetomek = SMOTETomek(random_state=0)
X_train_smotetomek, y_train_smotetomek = smotetomek.fit_resample(X_train, y_train)

**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [ ]:
# Logistic Regression with strategy = One vs. Rest
log_reg = LogisticRegression(solver="liblinear")
ovr_model = OneVsRestClassifier(log_reg)

In [ ]:
# SMOTE training set
ovr_model.fit(X_train_smote, y_train_smote)
ovr_predictions_smote = ovr_model.predict(X_test)
print(classification_report(y_test, ovr_predictions_smote))

In [ ]:
# SMOTENC training set
ovr_model.fit(X_train_smotenc, y_train_smotenc)
ovr_predictions_smotenc = ovr_model.predict(X_test)
print(classification_report(y_test, ovr_predictions_smotenc))

In [ ]:
# SMOTE-Tomek traning set
ovr_model.fit(X_train_smotetomek, y_train_smotetomek)
ovr_predictions_smotetomek = ovr_model.predict(X_test)
print(classification_report(y_test, ovr_predictions_smotetomek))

In [ ]:
# No resampling techniques applied
ovr_model.fit(X_train, y_train)
ovr_predictions = ovr_model.predict(X_test)
print(classification_report(y_test, ovr_predictions))

In [ ]:
def plot_roc(ax, X_train, y_train, X_test, y_test, title):
    clf = SVC(kernel='linear',probability=True)
    clf.fit(X_train, y_train)
    y_test_pred = clf.predict_proba(X_test)[:,1]
    fpr, tpr, thresh = metrics.roc_curve(y_test, y_test_pred)
    auc = metrics.roc_auc_score(y_test, y_test_pred)
    ax.plot(fpr,tpr,label=f"{title} AUC={auc:.3f}")

    ax.set_title('ROC Curve')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(loc=0)

In [ ]:
# SMOTE, SMOTENC, SMOTE-Tomek, no resampling
fig,ax = plt.subplots(1,1,figsize=(8,5))
plot_roc(ax, X_train_pca, y_train, X_test_pca, y_test, 'Data with no resampling methods applied')
plot_roc(ax, X_train_ros, y_train_ros, X_test_pca, y_test, 'SMOTE Oversampled Data')
plot_roc(ax, X_train_smote, y_train_smote, X_test_pca, y_test, 'SMOTENC Oversampled Data')
plot_roc(ax, X_train_adasyn, y_train_adasyn, X_test_pca, y_test, 'SMOTE-Tomek resampled Data');